# C1: NPPact Monthly Reconstruction

Reconstruct the monthly NPPact series from MODIS monthly PSN fractions and annual NPP.
- Method: NPPact(month) = NPP_annual × (PSN_month / PSN_annual_sum)
- Input: A1 monthly PSN + A2 annual NPP (aligned)
- Output: monthly NPPact → `interim/`

In [ ]:
import yaml
from pathlib import Path

import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd
from rasterio.enums import Resampling
import rioxarray
from geocube.api.core import make_geocube
import matplotlib.pyplot as plt

import sys
sys.path.append(str(Path('../src')))
from grid_utils import create_target_grid

with open(Path('../config.yaml')) as f:
    cfg = yaml.safe_load(f)

In [ ]:
for k, v in cfg['paths'].items():
    cfg["paths"][k] = Path(v)#.expanduser().resolve()

cfg["data_root"] = Path(cfg["data_root"])

In [ ]:
da_day_in_month = xr.DataArray([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31], dims=["month"], coords={"month": np.arange(1, 13)})

In [ ]:
da_template = create_target_grid()

# Reconstruct cropland NPPact

## Cropland fraction

In [ ]:
CROP_CLASSES = [12, 14]

In [ ]:
da_luc = (
    xr.open_dataarray(cfg["paths"]["gee_exports"] / f"luc/luc-500m.tif")
    .chunk({"band": 1, "x": 22000, "y": 22000})
    .assign_coords(band=range(2001, 2025))
    .rename({"band": "year"})
    .sel(year=slice(2010, 2024))
    .drop_attrs()
    .rio.write_crs("EPSG:4326")
    .astype(np.uint8)
)

In [ ]:
da_npp = (
    xr.open_dataarray(cfg["paths"]["gee_exports"] / f"npp/npp-500m.tif")
    .chunk({"band": 1, "x": 22000, "y": 22000})
    .assign_coords(band=range(2001, 2026))
    .rename({"band": "year"})
    .sel(year=slice(2010, 2024))
    .drop_attrs()
    .rio.write_crs("EPSG:4326")
)

In [ ]:
da_npp_exlude_crop = da_npp.where(~da_luc.isin(CROP_CLASSES)).coarsen(x=22, y=22, boundary="trim").mean().compute()
# .to_netcdf(cfg["paths"]["output"] / "npp_crop_monthly.nc")

In [ ]:
da_npp_exlude_crop.to_dataset(name="npp_exclude_crop").to_netcdf(
    cfg["paths"]["output"] / "npp_exclude_crop_annual.nc",
    encoding={"npp_exclude_crop": {"zlib": True, "complevel": 5}},
)

In [ ]:
da_crop_frac = da_luc.isin(CROP_CLASSES).astype(np.float32).coarsen(x=22, y=22, boundary="trim").mean().compute()

In [ ]:
da_crop_frac.to_dataset(name="crop_frac").to_netcdf(
    cfg["paths"]["output"] / "crop_frac.nc",
    encoding={"crop_frac": {"zlib": True, "complevel": 5}},
)

## Reconstruct annual NPPact

In [ ]:
year = 2015


In [ ]:
da_npp_exlude_crop = xr.open_dataarray(
    cfg["paths"]["output"] / "npp_exclude_crop_annual.nc")


In [ ]:
year = 2020

ds_crop_harv_unit = xr.open_dataset(cfg["paths"]["output"] / f"HANPPharv_crops_{year}.nc")
ds_crop_harv_total = ds_crop_harv_unit * da_template
da_crop_harv_total = ds_crop_harv_total.to_dataarray(dim="crop").sum(dim=["crop", "month"])
    
da_crop_frac = xr.open_dataarray(cfg["paths"]["output"] / "crop_frac.nc")\
    .sel(year=year)\
    .drop_vars("year")\
    .rio.write_crs("EPSG:4326")\
    .rio.reproject_match(da_template, resampling=Resampling.nearest) 
da_npp_exlude_crop_raw = xr.open_dataarray(cfg["paths"]["output"] / "npp_exclude_crop_annual.nc")\
    .sel(year=year)\
    .drop_vars("year")\
    .drop_attrs()\
    .rio.write_crs("EPSG:4326")\
    .rio.reproject_match(da_template, resampling=Resampling.nearest)\
    .pipe(lambda x: x*0.1) # to gC/m2

da_npp_exclude_crop = da_npp_exlude_crop_raw * da_template * (1-da_crop_frac)

In [ ]:
def compute_nppact_reconstruction(year):
    ds_crop_harv_unit = xr.open_dataset(cfg["paths"]["output"] / f"HANPPharv_crops_{year}.nc")
    ds_crop_harv_total = ds_crop_harv_unit * da_template
    da_crop_harv_total = ds_crop_harv_total.to_dataarray(dim="crop").sum(dim=["crop", "month"])

    da_crop_frac = xr.open_dataarray(cfg["paths"]["output"] / "crop_frac.nc")\
        .sel(year=year)\
        .drop_vars("year")\
        .rio.write_crs("EPSG:4326")\
        .rio.reproject_match(da_template, resampling=Resampling.nearest) 
    # da_crop_frac.plot.imshow()

    # da_npp_modis_annual = xr.open_dataarray(cfg["paths"]["gee_exports"] / "NPP_annual_2010_2025.tif")\
    #     .rename({"band": "year"})\
    #     .assign_coords(year=np.arange(2010, 2026))\
    #     .sel(year=year)\
    #     .drop_vars("year")\
    #     .drop_attrs()\
    #     .rio.write_crs("EPSG:4326")\
    #     .rio.reproject_match(da_template, resampling=Resampling.nearest)\
    #     .pipe(lambda x: x*0.1) # to gC/m2
    # da_npp_exclude_crop = da_npp_modis_annual * da_template * (1-da_crop_frac)
    
    da_npp_exlude_crop_raw = xr.open_dataarray(cfg["paths"]["output"] / "npp_exclude_crop_annual.nc")\
        .sel(year=year)\
        .drop_vars("year")\
        .drop_attrs()\
        .rio.write_crs("EPSG:4326")\
        .rio.reproject_match(da_template, resampling=Resampling.nearest)\
        .pipe(lambda x: x*0.1) # to gC/m2
    
    da_npp_exclude_crop = da_npp_exlude_crop_raw * da_template * (1-da_crop_frac)
    da_nppact_reconstruction = (da_crop_harv_total.fillna(0) + da_npp_exclude_crop.fillna(0)) / da_template.values
    return da_nppact_reconstruction

In [ ]:
lst_nppact_reconstruction = []
for year in range(2010, 2025):
    da_nppact_reconstruction = compute_nppact_reconstruction(year)
    lst_nppact_reconstruction.append(da_nppact_reconstruction)

In [ ]:
da_nppact_reconstruction_merge = xr.concat(lst_nppact_reconstruction, dim="year").assign_coords(year=np.arange(2010, 2025))
da_nppact_reconstruction_merge.to_dataset(name="nppact").to_netcdf(
    cfg["paths"]["output"] / "nppact_reconstruction.nc",
    encoding={"nppact": {"zlib": True, "complevel": 5}},
)

# Disaggregate to months

In [ ]:
def cal_nppact_monthly(year):
    da_npp_annual = xr.open_dataset(cfg["paths"]["output"] / "nppact_reconstruction.nc")["nppact"]\
        .sel(year=year)
    
    
    da_PSN = (
        xr.open_dataarray(cfg["paths"]["gee_exports"] / f"PSN_monthly_{year}.tif")\
        .rename({"band": "month"})\
        * da_day_in_month)\
        .drop_attrs()\
        .rename({"month": "time"})\
        .assign_coords(time=pd.date_range(f"{year}-01-01", f"{year}-12-31", freq="ME"))\
        .rio.write_crs("EPSG:4326")\
        .rio.reproject_match(da_template, resampling=Resampling.nearest)

    da_PSN_annual = da_PSN.sum("time")
    da_NPP_monthly = (da_PSN / da_PSN_annual) * da_npp_annual
    return da_NPP_monthly

In [ ]:
year = 2020
da_NPP_monthly = cal_nppact_monthly(year)

In [ ]:
lst_da_act = []
for year in range(2010, 2025):
    da_harv = cal_nppact_monthly(year)
    lst_da_act.append(da_harv)
da_act_merge = xr.concat(lst_da_act, dim='time')

In [ ]:
da_act_merge\
    .to_dataset(name='NPPact')\
    .to_netcdf(
        cfg["paths"]["output"] / 'monthly/NPPact.nc',
        encoding={'NPPact': {'zlib': True, 'complevel': 5}}
    )